## In this notebook we  -

#### 1. Create dummy or indicator features for categorical variables
#### 2. Standardize the magnitude of numeric features using a scaler
#### 3. Split your data into testing and training datasets



In our previous Step of EDA computation, we evaluated HBM theory through some selected feature variables and found that there are some good predictors of our target variable. However, in our analysis, we explicitly selected some variables by definition and this is not a comprehensive and complete list of variables which could have been in each of the categories of HBM model.

Here we try to engineer our existing features i.e make most of all of them such that a model can utilize it and then we categorize the most important ones according to HBM theory

### 1. Load data from last step of EDA

In [1]:
import pandas as pd
import numpy as np


In [2]:
import pickle

# Replace '002-eda_final.pkl' with the actual filename if different
with open('../data/processed/002-eda_final.pkl', 'rb') as f:
    df = pickle.load(f)

df.head() # Display the first few rows of the DataFrame to verify successful loading


,DISPCODE,GENHLTH,PHYSHLTH,MENTHLTH,HLTHPLN1,PERSDOC2,MEDCOST,CHECKUP1,CVDINFR4,CVDCRHD4,...,_EDUCAG,_INCOMG,_RFSMOK3,DRNKANY5,_RFBING5,_TOTINDA,_RFSEAT3,YEAR,DIABETES,_AGE65YR
0,1200,5,3,4,1,1,2,1,2,2,...,2,2,0,2,1,2,1,2015,2,1
1,1100,3,1,1,2,1,1,4,2,2,...,4,1,1,2,1,1,2,2015,2,1
3,1100,5,4,4,1,2,1,1,2,2,...,2,5,0,2,1,2,1,2015,2,1
4,1100,5,4,1,1,1,2,1,2,2,...,3,5,0,2,1,2,1,2015,2,1
5,1100,2,1,1,1,1,2,1,2,2,...,1,4,0,2,1,1,1,2015,2,2


In [3]:
df.describe()  # Display summary statistics of the DataFrame

,CHILDREN,WEIGHT2,HEIGHT3
count,2311512.0,2.311512e+06,2.311512e+06
mean,0.513319,8.080105e+01,1.692953e+00
std,0.996833,2.094946e+01,1.040322e-01
min,0.0,1.000000e+00,7.366000e-01
25%,0.0,6.577200e+01,1.625600e+00
50%,0.0,7.801920e+01,1.676400e+00
75%,1.0,9.072000e+01,1.778000e+00
max,5.0,3.000000e+02,3.170000e+00


### 2. One-hot encode categorical columns

In [4]:
cat_cols = df.select_dtypes(include=['category', 'object']).columns.tolist()
print(f"Categorical columns: {cat_cols}")


Categorical columns: ['DISPCODE', 'GENHLTH', 'PHYSHLTH', 'MENTHLTH', 'HLTHPLN1', 'PERSDOC2', 'MEDCOST', 'CHECKUP1', 'CVDINFR4', 'CVDCRHD4', 'CVDSTRK3', 'ASTHMA3', 'CHCSCNCR', 'CHCOCNCR', 'CHCCOPD1', 'HAVARTH3', 'ADDEPEV2', 'CHCKIDNY', 'DIABETE3', 'SEX', 'MARITAL', 'RENTHOM1', 'VETERAN3', 'QLACTLM2', 'USEEQUIP', 'EXERANY2', 'PNEUVAC3', 'HIVTST6', '_RFHLTH', '_HCVU651', '_LTASTH1', '_CASTHM1', '_ASTHMS1', '_DRDXAR1', '_RFBMI5', '_CHLDCNT', '_EDUCAG', '_INCOMG', '_RFSMOK3', 'DRNKANY5', '_RFBING5', '_TOTINDA', '_RFSEAT3', 'YEAR', 'DIABETES', '_AGE65YR']


In [5]:
#drop the target column from the list of categorical columns
cat_cols = [col for col in cat_cols if col != '_RFSMOK3']
print(f"Final Categorical cols: {cat_cols}")

Final Categorical cols: ['DISPCODE', 'GENHLTH', 'PHYSHLTH', 'MENTHLTH', 'HLTHPLN1', 'PERSDOC2', 'MEDCOST', 'CHECKUP1', 'CVDINFR4', 'CVDCRHD4', 'CVDSTRK3', 'ASTHMA3', 'CHCSCNCR', 'CHCOCNCR', 'CHCCOPD1', 'HAVARTH3', 'ADDEPEV2', 'CHCKIDNY', 'DIABETE3', 'SEX', 'MARITAL', 'RENTHOM1', 'VETERAN3', 'QLACTLM2', 'USEEQUIP', 'EXERANY2', 'PNEUVAC3', 'HIVTST6', '_RFHLTH', '_HCVU651', '_LTASTH1', '_CASTHM1', '_ASTHMS1', '_DRDXAR1', '_RFBMI5', '_CHLDCNT', '_EDUCAG', '_INCOMG', 'DRNKANY5', '_RFBING5', '_TOTINDA', '_RFSEAT3', 'YEAR', 'DIABETES', '_AGE65YR']


In [6]:
df.columns  # Display the columns of the DataFrame to see the new features created by one-hot encoding

Index(['DISPCODE', 'GENHLTH', 'PHYSHLTH', 'MENTHLTH', 'HLTHPLN1', 'PERSDOC2',
       'MEDCOST', 'CHECKUP1', 'CVDINFR4', 'CVDCRHD4', 'CVDSTRK3', 'ASTHMA3',
       'CHCSCNCR', 'CHCOCNCR', 'CHCCOPD1', 'HAVARTH3', 'ADDEPEV2', 'CHCKIDNY',
       'DIABETE3', 'SEX', 'MARITAL', 'RENTHOM1', 'VETERAN3', 'CHILDREN',
       'WEIGHT2', 'HEIGHT3', 'QLACTLM2', 'USEEQUIP', 'EXERANY2', 'PNEUVAC3',
       'HIVTST6', '_RFHLTH', '_HCVU651', '_LTASTH1', '_CASTHM1', '_ASTHMS1',
       '_DRDXAR1', '_RFBMI5', '_CHLDCNT', '_EDUCAG', '_INCOMG', '_RFSMOK3',
       'DRNKANY5', '_RFBING5', '_TOTINDA', '_RFSEAT3', 'YEAR', 'DIABETES',
       '_AGE65YR'],
      dtype='object')

In [7]:
df_encoded = df.copy()  # Create a copy of the DataFrame to avoid modifying the original

In [8]:
#one-hot encode categorical columns
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

In [9]:
df_encoded.shape  # Display the shape of the DataFrame to see how many features we have after encoding

(2311512, 113)

### 3. Standardize numerical columns

In [10]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numerical columns: {num_cols}")

Numerical columns: ['CHILDREN', 'WEIGHT2', 'HEIGHT3']


In [11]:
#Standardize numerical columns
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df_encoded[num_cols] = scaler.fit_transform(df_encoded[num_cols])
# Save the processed DataFrame to a new pickle file
output_file = '../data/processed/003-feature-engineered.pkl'
with open(output_file, 'wb') as f:
    pickle.dump(df_encoded, f)
print(f"Processed DataFrame saved to {output_file}")

Processed DataFrame saved to ../data/processed/003-feature-engineered.pkl


### 4. Split the dataset into train and test set

##### a. Split with One-hot encoding and standardized numeric columns

In [12]:
from sklearn.model_selection import train_test_split

# Example: Let's assume '_RFSMOK3_1' is the target variable for classification
target_col = '_RFSMOK3'
X_encoded = df_encoded.drop(columns=[target_col])
y_encoded = df_encoded[target_col]

X_train_encoded, X_test_encoded, y_train_encoded, y_test_encoded = train_test_split(X_encoded, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
print(f"Train shape: {X_train_encoded.shape}, Test shape: {X_test_encoded.shape}")

Train shape: (1849209, 112), Test shape: (462303, 112)


##### b. Split orginal dataset without One-hot encoding or standardization

In [13]:
from sklearn.model_selection import train_test_split

# Example: Let's assume '_RFSMOK3_1' is the target variable for classification
target_col = '_RFSMOK3'
X = df.drop(columns=[target_col])
y = df_encoded[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Train shape: (1849209, 48), Test shape: (462303, 48)


### 5. Save the test and train sets into pickle files

#### a. Save encoded X_train/X_test

In [14]:
import os

# Ensure the directory exists
os.makedirs('../data/modeling/', exist_ok=True)

# Save the splits
with open('../data/modeling/data_splits_encoded.pkl', 'wb') as f:
    pickle.dump({
        'X_train': X_train_encoded,
        'X_test': X_test_encoded, 
        'y_train': y_train_encoded,
        'y_test': y_test_encoded
    }, f)

#### b. Save original X_train/X_test

In [15]:
# Save the splits
with open('../data/modeling/data_splits.pkl', 'wb') as f:
    pickle.dump({
        'X_train': X_train,
        'X_test': X_test, 
        'y_train': y_train,
        'y_test': y_test
    }, f)

## Lets take a look at our datasets one last time

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2311512 entries, 0 to 2380046
Data columns (total 49 columns):
 #   Column    Dtype   
---  ------    -----   
 0   DISPCODE  category
 1   GENHLTH   category
 2   PHYSHLTH  category
 3   MENTHLTH  category
 4   HLTHPLN1  category
 5   PERSDOC2  category
 6   MEDCOST   category
 7   CHECKUP1  category
 8   CVDINFR4  category
 9   CVDCRHD4  category
 10  CVDSTRK3  category
 11  ASTHMA3   category
 12  CHCSCNCR  category
 13  CHCOCNCR  category
 14  CHCCOPD1  category
 15  HAVARTH3  category
 16  ADDEPEV2  category
 17  CHCKIDNY  category
 18  DIABETE3  category
 19  SEX       category
 20  MARITAL   category
 21  RENTHOM1  category
 22  VETERAN3  category
 23  CHILDREN  Int64   
 24  WEIGHT2   float64 
 25  HEIGHT3   float64 
 26  QLACTLM2  category
 27  USEEQUIP  category
 28  EXERANY2  category
 29  PNEUVAC3  category
 30  HIVTST6   category
 31  _RFHLTH   category
 32  _HCVU651  category
 33  _LTASTH1  category
 34  _CASTHM1  category
 35 

In [17]:
X_train.info()  # Display the first few rows of the training set to verify successful encoding

<class 'pandas.core.frame.DataFrame'>
Index: 1849209 entries, 1395283 to 755824
Data columns (total 48 columns):
 #   Column    Dtype   
---  ------    -----   
 0   DISPCODE  category
 1   GENHLTH   category
 2   PHYSHLTH  category
 3   MENTHLTH  category
 4   HLTHPLN1  category
 5   PERSDOC2  category
 6   MEDCOST   category
 7   CHECKUP1  category
 8   CVDINFR4  category
 9   CVDCRHD4  category
 10  CVDSTRK3  category
 11  ASTHMA3   category
 12  CHCSCNCR  category
 13  CHCOCNCR  category
 14  CHCCOPD1  category
 15  HAVARTH3  category
 16  ADDEPEV2  category
 17  CHCKIDNY  category
 18  DIABETE3  category
 19  SEX       category
 20  MARITAL   category
 21  RENTHOM1  category
 22  VETERAN3  category
 23  CHILDREN  Int64   
 24  WEIGHT2   float64 
 25  HEIGHT3   float64 
 26  QLACTLM2  category
 27  USEEQUIP  category
 28  EXERANY2  category
 29  PNEUVAC3  category
 30  HIVTST6   category
 31  _RFHLTH   category
 32  _HCVU651  category
 33  _LTASTH1  category
 34  _CASTHM1  category